In [10]:
import pickle

with open('../image_processing/dataset_two_class.pkl', 'rb') as file:
    data=pickle.load(file)

In [66]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Reduces TF logging to errors only

In [11]:
X=data['processed_image_data']
y=data['processed_class_data']

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test=train_test_split(X, y, test_size=0.2, random_state=42)
(x_train, y_train), (x_test, y_test)=(X_train, Y_train), (X_test, Y_test)

In [15]:
x_train=x_train.reshape(-1, 28, 28, 1)
x_test=x_test.reshape(-1, 28, 28, 1)

In [16]:
print(x_train.shape)
print(x_test.shape)

(1456, 28, 28, 1)
(365, 28, 28, 1)


In [26]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense

model=Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(28, 28, 1)),
    MaxPool2D((2,2)),

    Conv2D(32, (3,3), activation='relu'),
    MaxPool2D((2,2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

In [34]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [35]:
history=model.fit(x_train, y_train, epochs=15, batch_size=32, validation_data=(x_test, y_test))

Epoch 1/15


46/46 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.9663 - loss: 0.0882 - val_accuracy: 0.9699 - val_loss: 0.0737
Epoch 2/15
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9760 - loss: 0.0725 - val_accuracy: 0.9671 - val_loss: 0.0648
Epoch 3/15
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9766 - loss: 0.0716 - val_accuracy: 0.9644 - val_loss: 0.0784
Epoch 4/15
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9780 - loss: 0.0595 - val_accuracy: 0.9726 - val_loss: 0.0769
Epoch 5/15
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9705 - loss: 0.0703 - val_accuracy: 0.9671 - val_loss: 0.0809
Epoch 6/15
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.9705 - loss: 0.0788 - val_accuracy: 0.9781 - val_loss: 0.0599
Epoch 7/15
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9849 - loss: 0.0479 - val_accuracy: 0.9644 - val_loss: 0.0685
Epoch 8/15
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9849 - loss: 0.0432 - val_accuracy: 0.9726 - val_loss: 0.

In [37]:
test_loss, test_acc=model.evaluate(x_test, y_test)
print('test_loss: ',test_loss)
print('test_acc: ', test_acc)

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9726 - loss: 0.0754
test_loss:  0.07541599124670029
test_acc:  0.9726027250289917


In [65]:
import cv2
import numpy as np
img1='../image_processing/Skin cancer ISIC The International Skin Imaging Collaboration/images/ISIC_0000000.jpg'
img0='../image_processing/healthy_skin/3fdg6hg.jpg'

img_can=cv2.imread(img1)
img_hel=cv2.imread(img0)

if img_can is not None:
    resize_can=cv2.resize(img_can, (28, 28), interpolation=cv2.INTER_AREA)
    gray=cv2.cvtColor(resize_can, cv2.COLOR_BGR2GRAY)
    normalized=gray.astype('float32')/255.0
    val=np.array(normalized)
    val=val.reshape(-1, 28, 28)
    prop=model.predict(val)
    if prop > 0.5:
        print('having skin cancer')
    else:
        print('not having skin cancer')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step
having skin cancer


In [64]:
file_path='cnn_model.pkl'

with open(file_path, 'wb') as file:
    pickle.dump(model, file)
